## Installing the SageMaker SDK

In [ ]:
%pip install -q --upgrade -r requirements.txt

%pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops

%pip install --no-cache-dir --force-reinstall "sagemaker>=2.245.0,<3"

In [ ]:
import base64
import io
import json 
import shutil
import sys
import tarfile


import time 

from urllib.parse import urlparse

import requests
import boto3 
import numpy as np 
import sagemaker

In [ ]:
%pip install -q "pyarrow>=14.0" "datasets>=2.16"

In [ ]:
from datasets import load_dataset
from sagemaker import image_uris
from sagemaker.huggingface import HuggingFace, HuggingFaceModel
from sagemaker.model_monitor import DataCaptureConfig 
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

In [ ]:
sys.path.append('scripts')

session = sagemaker.Session()
boto_session = boto3.Session()
REGION = boto_session.region_name
ROLE = sagemaker.get_execution_role()
BUCKET = 'cxr14-tfstate-c70a9673'

print({'region': REGION, 'bucket': BUCKET, 'role': ROLE})

In [ ]:
import matplotlib 
import matplotlib.pyplot as plt

In [ ]:
from datasets import load_dataset_builder
HF_DATASET_ID   = 'BahaaEldin0/NIH-Chest-Xray-14'
b = load_dataset_builder(HF_DATASET_ID)
print(b.info.features)   # shows the real column types
print(b.info.splits)     # row counts per split

In [ ]:
import os
import numpy as np
import pandas as pd
from huggingface_hub import HfApi
from datasets import load_dataset_builder

HF_DATASET_ID  = "BahaaEldin0/NIH-Chest-Xray-14"
LABEL_COLUMN   = "label"
PATIENT_AGE    = "Patient Age"
PATIENT_GENDER = "Patient Gender"
VIEW_POSITION  = "View Position"
PATIENT_ID     = "Patient ID"
TRAIN_SPLIT, VALID_SPLIT, TEST_SPLIT = "train", "valid", "test"

In [ ]:
files = [f for f in HfApi().list_repo_files(HF_DATASET_ID, repo_type="dataset")
         if f.endswith(".parquet")]
print(len(files))
for f in files:
    print(f)

In [ ]:
%%time
df = pd.read_parquet(f"hf://datasets/{HF_DATASET_ID}/{files[0]}", columns=cols)
print(len(files), "files;", len(df), "rows in the first")

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

def split_of(f):
    for s in (TRAIN_SPLIT, VALID_SPLIT, TEST_SPLIT):
        if s in f:
            return s

def read_one(f):
    df = pd.read_parquet(f"hf://datasets/{HF_DATASET_ID}/{f}", columns=cols)
    df["split"] = split_of(f)
    df["file"] = f
    return df

with ThreadPoolExecutor(max_workers=8) as ex:
    parts = list(tqdm(ex.map(read_one, files), total=len(files)))

meta = pd.concat(parts, ignore_index=True)
meta.to_parquet("metadata.parquet")     # cachING : reload this next time instead of refetching
print(meta["split"].value_counts())

In [ ]:
os.makedirs("data", exist_ok=True)
meta["row_in_file"] = meta.groupby("file").cumcount()   # position of each row inside 
meta.to_parquet("data/metadata.parquet")

In [ ]:
g = {s: set(meta.loc[meta["split"] == s, PATIENT_ID]) for s in ("train", "valid", "test")}

print("patients per split:", {s: len(v) for s, v in g.items()})
print("train ∩ valid:", len(g["train"] & g["valid"]))
print("train ∩ test: ", len(g["train"] & g["test"]))
print("valid ∩ test: ", len(g["valid"] & g["test"]))


In [ ]:
print(meta[PATIENT_AGE].describe())
print("ages > 100:", (meta[PATIENT_AGE] > 100).sum())

print(meta[PATIENT_GENDER].value_counts())
print(meta[VIEW_POSITION].value_counts())

train = meta[meta["split"] == "train"]
label_names = sorted(set(train[LABEL_COLUMN].explode().dropna()) - {"No Finding"})
print(len(label_names), label_names)                 # expect 14
print(train[LABEL_COLUMN].explode().value_counts())

In [ ]:
meta["age_clean"] = meta[PATIENT_AGE].where(meta[PATIENT_AGE].between(1, 100))   # invalid ages become NaN
print("ages set to missing:", meta["age_clean"].isna().sum())

sha = HfApi().dataset_info(HF_DATASET_ID).sha
print("dataset revision:", sha)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
groups = meta[PATIENT_ID]

# 80% train+valid / 20% test, keeping each patient entirely on one side
tv_idx, te_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED).split(meta, groups=groups))
trainval = meta.iloc[tv_idx]

# split train+valid again: 12.5% of 80% is 10% overall
tr_idx, va_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=SEED)
                      .split(trainval, groups=trainval[PATIENT_ID]))

meta["new_split"] = "train"
meta.loc[trainval.index[va_idx], "new_split"] = "valid"
meta.loc[meta.index[te_idx], "new_split"] = "test"

# Hard check: no patient in more than one split
n_splits_per_patient = meta.groupby(PATIENT_ID)["new_split"].nunique()
assert (n_splits_per_patient == 1).all(), "leakage!"
print(meta["new_split"].value_counts())

In [ ]:
# Labels spread sensibly across the new splits?
Yall = meta[LABEL_COLUMN].map(lambda ls: [l for l in ls if l != "No Finding"])
prev = (pd.get_dummies(Yall.explode()).groupby(Yall.explode().index).max()
          .join(meta["new_split"]).groupby("new_split").sum())
print(prev[label_names].T)     # rows are findings, columns are splits

# Save ONLY the patient 
os.makedirs("configs", exist_ok=True)
(meta[[PATIENT_ID, "new_split"]].drop_duplicates()
     .sort_values(PATIENT_ID)
     .to_csv("configs/patient_split_v1.csv", index=False))

meta.to_parquet("data/metadata.parquet")   # full cache stays out of Git